<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day26_AI_Helpdesk_API_Redis_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install fastapi uvicorn requests redis


In [2]:
import fastapi, requests, redis
print('FastAPI:', fastapi.__version__)
print('Requests:', requests.__version__)
print('Redis client imported successfully')

FastAPI: 0.141.1
Requests: 2.32.4
Redis client imported successfully


In [3]:
ticket_database = {
    "TKT5001": {
        "employee_id": "EMP1001",
        "employee_name": "Amit Sharma",
        "department": "Finance",
        "issue": "Finance Reporting Access Denied",
        "priority": "High",
        "status": "Open",
        "approval": "Pending"
    },
    "TKT5002": {
        "employee_id": "EMP1002",
        "employee_name": "Neha Verma",
        "department": "HR",
        "issue": "Payroll Portal Login Issue",
        "priority": "Medium",
        "status": "In Progress",
        "approval": "Not Required"
    }
}
print(ticket_database["TKT5001"])

{'employee_id': 'EMP1001', 'employee_name': 'Amit Sharma', 'department': 'Finance', 'issue': 'Finance Reporting Access Denied', 'priority': 'High', 'status': 'Open', 'approval': 'Pending'}


In [4]:
assert ticket_database["TKT5001"]["status"] == "Open"
assert ticket_database["TKT5001"]["approval"] == "Pending"
print("Ticket source data verified.")

Ticket source data verified.


In [5]:
import redis
from google.colab import userdata

redis_client = redis.Redis(
    host="lenient-snazzy-mintish-71890.db.redis.io",
    port=16413,
    username="default",
    password=userdata.get("REDIS_PASSWORD"),
    decode_responses=True
)

print("Redis connection:", redis_client.ping())

Redis connection: True


In [7]:
import redis
from google.colab import userdata

redis_client = redis.Redis(
    host="lenient-snazzy-mintish-71890.db.redis.io",
    port=16413,
    username="default",
    password=userdata.get("REDIS_PASSWORD"),
    decode_responses=True
)

print("Redis connection:", redis_client.ping())

Redis connection: True


In [8]:
redis_client.set("employee:EMP1001", "Amit Sharma")

employee = redis_client.get("employee:EMP1001")

print("Employee:", employee)

Employee: Amit Sharma


In [9]:
# Store employee information in Redis

redis_client.set("employee:EMP1001", "Amit Sharma")
redis_client.set("ticket:TKT5001", "Finance Reporting Access Denied")

# Retrieve stored information

employee = redis_client.get("employee:EMP1001")
ticket = redis_client.get("ticket:TKT5001")

print("Employee:", employee)
print("Ticket:", ticket)

Employee: Amit Sharma
Ticket: Finance Reporting Access Denied


In [10]:
import json

ticket_data = {
    "ticket_id": "TKT5001",
    "employee_id": "EMP1001",
    "issue": "Finance Reporting Access Denied",
    "priority": "High",
    "status": "Open"
}

redis_client.set(
    "ticket:TKT5001",
    json.dumps(ticket_data)
)

retrieved_ticket = json.loads(
    redis_client.get("ticket:TKT5001")
)

print(json.dumps(retrieved_ticket, indent=4))

{
    "ticket_id": "TKT5001",
    "employee_id": "EMP1001",
    "issue": "Finance Reporting Access Denied",
    "priority": "High",
    "status": "Open"
}


In [11]:
!pip install fastapi uvicorn httpx -q

In [12]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import json

app = FastAPI(
    title="IT Helpdesk API",
    description="Day 26 - Redis and FastAPI Integration",
    version="1.0"
)

# Define the ticket structure
class Ticket(BaseModel):
    ticket_id: str
    employee_id: str
    issue: str
    priority: str
    status: str

# Health check endpoint
@app.get("/")
def home():
    return {
        "message": "IT Helpdesk API is running",
        "redis_status": "Connected"
    }

# Store ticket in Redis
@app.post("/tickets")
def create_ticket(ticket: Ticket):

    redis_client.set(
        f"ticket:{ticket.ticket_id}",
        ticket.model_dump_json()
    )

    return {
        "message": "Ticket stored successfully",
        "ticket_id": ticket.ticket_id
    }

# Retrieve ticket from Redis
@app.get("/tickets/{ticket_id}")
def get_ticket(ticket_id: str):

    data = redis_client.get(f"ticket:{ticket_id}")

    if data is None:
        raise HTTPException(
            status_code=404,
            detail="Ticket not found"
        )

    return json.loads(data)

In [13]:
from fastapi.testclient import TestClient

client = TestClient(app)

# Test the home endpoint
response = client.get("/")

print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'message': 'IT Helpdesk API is running', 'redis_status': 'Connected'}


In [14]:
ticket_payload = {
    "ticket_id": "TKT5002",
    "employee_id": "EMP1002",
    "issue": "Unable to access HR Portal",
    "priority": "Medium",
    "status": "Open"
}

response = client.post("/tickets", json=ticket_payload)

print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'message': 'Ticket stored successfully', 'ticket_id': 'TKT5002'}


In [15]:
response = client.get("/tickets/TKT5002")

print("Status:", response.status_code)
print("Ticket Details:")

print(json.dumps(response.json(), indent=4))

Status: 200
Ticket Details:
{
    "ticket_id": "TKT5002",
    "employee_id": "EMP1002",
    "issue": "Unable to access HR Portal",
    "priority": "Medium",
    "status": "Open"
}


In [16]:
def create_helpdesk_ticket(ticket_data):
    response = client.post(
        "/tickets",
        json=ticket_data
    )

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()


def retrieve_helpdesk_ticket(ticket_id):
    response = client.get(
        f"/tickets/{ticket_id}"
    )

    if response.status_code == 404:
        return {"error": "Ticket not found"}

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()

In [17]:
new_ticket = {
    "ticket_id": "TKT5003",
    "employee_id": "EMP1003",
    "issue": "Unable to access Power BI dashboard",
    "priority": "High",
    "status": "Open"
}

# Create ticket
create_result = create_helpdesk_ticket(new_ticket)

print("CREATE RESULT:")
print(create_result)

# Retrieve ticket
ticket_result = retrieve_helpdesk_ticket("TKT5003")

print("\nRETRIEVED TICKET:")
print(ticket_result)

CREATE RESULT:
{'message': 'Ticket stored successfully', 'ticket_id': 'TKT5003'}

RETRIEVED TICKET:
{'ticket_id': 'TKT5003', 'employee_id': 'EMP1003', 'issue': 'Unable to access Power BI dashboard', 'priority': 'High', 'status': 'Open'}


In [18]:
import json

def get_ticket_from_cache(ticket_id):
    cached_data = redis_client.get(ticket_id)
    if cached_data:
        print("Cache HIT:", ticket_id)
        return json.loads(cached_data)
    print("Cache MISS:", ticket_id)
    return None

def save_ticket_to_cache(ticket_id, ticket_data):
    redis_client.setex(ticket_id, 300, json.dumps(ticket_data))
    print("Saved to Redis for 300 seconds:", ticket_id)

In [19]:
save_ticket_to_cache("TKT5001", ticket_database["TKT5001"])
cached_ticket = get_ticket_from_cache("TKT5001")
print(cached_ticket)

/tmp/ipykernel_2372/2292977735.py:12: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  redis_client.setex(ticket_id, 300, json.dumps(ticket_data))


Saved to Redis for 300 seconds: TKT5001
Cache HIT: TKT5001
{'employee_id': 'EMP1001', 'employee_name': 'Amit Sharma', 'department': 'Finance', 'issue': 'Finance Reporting Access Denied', 'priority': 'High', 'status': 'Open', 'approval': 'Pending'}


In [20]:
from fastapi.testclient import TestClient
client = TestClient(app)

response = client.get("/")
print("Status:", response.status_code)
print("Body:", response.json())

Status: 200
Body: {'message': 'IT Helpdesk API is running', 'redis_status': 'Connected'}


In [21]:
response = client.get("/ticket/TKT5001")
print("Status:", response.status_code)
print("Ticket:", response.json())

Status: 404
Ticket: {'detail': 'Not Found'}


In [22]:
print("First request")
r1 = client.get("/ticket/TKT5002")
print(r1.status_code, r1.json())

print("\\nSecond request")
r2 = client.get("/ticket/TKT5002")
print(r2.status_code, r2.json())

First request
404 {'detail': 'Not Found'}
\nSecond request
404 {'detail': 'Not Found'}


In [23]:
missing_response = client.get("/ticket/TKT9999")
print("Status:", missing_response.status_code)
print("Body:", missing_response.json())

Status: 404
Body: {'detail': 'Not Found'}


In [24]:
def fetch_ticket_api(ticket_id: str):
    response = client.get(f"/ticket/{ticket_id}")

    if response.status_code == 404:
        return {"error": "Ticket not found"}

    response.raise_for_status()
    return response.json()

ticket_result = fetch_ticket_api("TKT5001")
print(ticket_result)

{'error': 'Ticket not found'}


In [25]:
print(fetch_ticket_api("TKT5001"))
print(fetch_ticket_api("TKT9999"))

{'error': 'Ticket not found'}
{'error': 'Ticket not found'}


In [27]:
# Check whether the ticket exists in Redis

print("TKT5001 exists:",
      redis_client.exists("ticket:TKT5001"))

print("TKT5002 exists:",
      redis_client.exists("ticket:TKT5002"))

print("TKT5003 exists:",
      redis_client.exists("ticket:TKT5003"))

TKT5001 exists: 1
TKT5002 exists: 1
TKT5003 exists: 1


In [28]:
import inspect

# 1. Read the ticket directly from Redis
print("DIRECT REDIS DATA:")
print(redis_client.get("ticket:TKT5001"))

# 2. Inspect the API retrieval function
print("\nFETCH FUNCTION:")
print(inspect.getsource(fetch_ticket_api))

# 3. Test the FastAPI endpoint directly
print("\nFASTAPI RESPONSE:")
response = client.get("/tickets/TKT5001")

print("Status code:", response.status_code)
print("Response:", response.text)

DIRECT REDIS DATA:
{"ticket_id": "TKT5001", "employee_id": "EMP1001", "issue": "Finance Reporting Access Denied", "priority": "High", "status": "Open"}

FETCH FUNCTION:
def fetch_ticket_api(ticket_id: str):
    response = client.get(f"/ticket/{ticket_id}")

    if response.status_code == 404:
        return {"error": "Ticket not found"}

    response.raise_for_status()
    return response.json()


FASTAPI RESPONSE:
Status code: 200
Response: {"ticket_id":"TKT5001","employee_id":"EMP1001","issue":"Finance Reporting Access Denied","priority":"High","status":"Open"}


In [29]:
def fetch_ticket_api(ticket_id: str):

    response = client.get(f"/tickets/{ticket_id}")

    if response.status_code == 404:
        return {"error": "Ticket not found"}

    response.raise_for_status()

    return response.json()

In [30]:
print("Existing ticket:")
print(fetch_ticket_api("TKT5001"))

print("\nNon-existing ticket:")
print(fetch_ticket_api("TKT9999"))

Existing ticket:
{'ticket_id': 'TKT5001', 'employee_id': 'EMP1001', 'issue': 'Finance Reporting Access Denied', 'priority': 'High', 'status': 'Open'}

Non-existing ticket:
{'error': 'Ticket not found'}


In [32]:
!pip install -q groq

In [33]:
from groq import Groq
from google.colab import userdata

groq_client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

print("Groq client initialized successfully!")

Groq client initialized successfully!


In [35]:
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain the role of Redis and FastAPI in an IT Helpdesk system in three bullet points."
        }
    ]
)

print(response.choices[0].message.content)

- **FastAPI**: Acts as the lightweight, async web framework that exposes REST/GraphQL endpoints for ticket submission, status updates, and user authentication, delivering high‑performance, type‑safe API interactions for both internal tools and external clients.

- **Redis**: Serves as the in‑memory data store for real‑time features such as a message queue (for notifying agents of new tickets), session caching, and a leaderboard of agent performance metrics, ensuring sub‑millisecond latency and horizontal scalability.

- **Integrated Workflow**: FastAPI receives an incoming ticket, pushes it to a Redis queue; worker processes consume the queue, apply business logic, update ticket status in a persistent database, and push notifications back through Redis Pub/Sub to trigger instant UI refreshes or email alerts. This architecture keeps the system responsive, fault‑tolerant, and easily extensible.


In [38]:
def generate_helpdesk_response(ticket_id):

    # Retrieve verified ticket details
    ticket_data = fetch_ticket_api(ticket_id)

    if "error" in ticket_data:
        return ticket_data["error"]

    # Retrieve employee information
    employee_id = ticket_data["employee_id"]
    employee_data = employee_records.get(employee_id, {})

    employee_name = employee_data.get(
        "employee_name", "Employee"
    )

    current_role = employee_data.get(
        "current_role", "Not available"
    )

    requested_role = employee_data.get(
        "requested_role", "Not available"
    )

    approval = employee_data.get(
        "manager_approval", "Not available"
    )

    prompt = f"""
    You are an IT Helpdesk Support Assistant.

    Prepare a professional and personalized ticket update.

    Verified Employee Information:
    Employee Name: {employee_name}
    Employee ID: {employee_id}
    Current Role: {current_role}
    Requested Role: {requested_role}
    Manager Approval: {approval}

    Verified Ticket Information:
    Ticket ID: {ticket_data['ticket_id']}
    Issue: {ticket_data['issue']}
    Priority: {ticket_data['priority']}
    Status: {ticket_data['status']}

    Instructions:
    1. Address the employee by name.
    2. Explain the current ticket status.
    3. Mention ticket priority.
    4. Explain that manager approval is pending.
    5. Never claim that access has been granted.
    6. Do not modify any employee records.
    7. Use only the supplied verified information.
    """

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [40]:
employee_records = {
    "EMP1001": {
        "employee_name": "Amit Sharma",
        "department": "Finance",
        "current_role": "Finance_Viewer",
        "requested_role": "Finance_Analyst",
        "manager_approval": "Pending"
    }
}

print("Employee records initialized successfully!")

print(employee_records["EMP1001"])

Employee records initialized successfully!
{'employee_name': 'Amit Sharma', 'department': 'Finance', 'current_role': 'Finance_Viewer', 'requested_role': 'Finance_Analyst', 'manager_approval': 'Pending'}


In [41]:
print(generate_helpdesk_response("TKT5001"))

Subject: Update on Ticket TKT5001 – Finance Reporting Access Denied

Hi Amit,

I wanted to give you a quick update on your request for a role change from Finance_Viewer to Finance_Analyst (Ticket ID: TKT5001).

- **Ticket Status**: Open  
- **Priority**: High  
- **Current Progress**: Your role upgrade request is currently awaiting approval from your manager.  
- **Next Steps**: Once manager approval is granted, we will proceed with the necessary permissions and inform you immediately.

Thank you for your patience. Please let me know if you need any additional information or if there is anything else I can assist you with.

Best regards,  
IT Helpdesk Support Team
